<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9_multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 9 (multiclass) — training the normalized hNPE–hNDE bridge

This is an **alternative** to `Exercise_9_Hybrid_NPE_NDE.ipynb`; the original exercise is not changed.  We keep exactly the same Gaussian simulator and the same defensive simulation design, but ask a different machine-learning question:

> Can one shared multiclass density-ratio model learn the posterior and likelihood residuals *together*, while normalization and Bayes compatibility are part of the training objective rather than only post-training checks?

The notebook contains two experiments.

1. **Nuisance marginalized implicitly.**  We omit $\alpha$ from the learned parameter and make a clean bridge ablation: both classifiers use
   $\mathcal L_{\rm CE}+\lambda_Z\mathcal L_{\rm normalization}$, while only the second adds $\lambda_B\mathcal L_{\rm bridge}$.
2. **Nuisance represented explicitly.**  We add $\alpha$ and train the full four-class objective, including posterior, conditional-nuisance, likelihood, and joint-posterior partitions.

**Scope.**  This first implementation jointly trains the shared residual discriminator around frozen NPE/NDE proposal flows.  It is not yet end-to-end joint optimization of the flows themselves: bridge gradients update the classifier logits only.

The analytic likelihood is used only after training for closure plots.  Every main figure is also exported as an editable standalone Matplotlib script for use in the ML paper.


In [ ]:
# ========================================================================
# Google Colab setup — safe to re-run; a no-op outside Colab.
# ========================================================================
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = True

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)
    run(sys.executable, "-m", "pip", "install", "-q", "nflows")
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
else:
    candidates = [Path.cwd(), Path.cwd() / "workshops" / "ml4hep_tifr_colab"]
    TUTORIAL_DIR = next(
        (candidate for candidate in candidates if (candidate / "utils_hnpe.py").exists()),
        None,
    )
    if TUTORIAL_DIR is None:
        raise FileNotFoundError(
            "Run from the repository root or workshops/ml4hep_tifr_colab."
        )
    if str(TUTORIAL_DIR.resolve()) not in sys.path:
        sys.path.insert(0, str(TUTORIAL_DIR.resolve()))


## What is actually being tested?

A balanced $K$-class classifier with shared logits $s_k(z)$ estimates

$$
  \log\frac{\pi_a(z)}{\pi_b(z)}=s_a(z)-s_b(z).
$$

Sharing logits is useful, but it also makes every pairwise log-ratio cycle vanish **algebraically**, even for an untrained network.  We therefore never use a cycle such as
$(s_S-s_P)+(s_P-s_L)+(s_L-s_S)=0$ as evidence of learned consistency.

Our non-tautological tests are instead:

- Does each residual integrate to one against its frozen proposal, condition by condition?
- After those conditional partitions are included, is the implied evidence independent of the parameter at fixed $x$?
- With an explicit nuisance, is the nuisance-marginal likelihood independent of the particular $\alpha$ used to reconstruct it?

All classifier classes occur once per anchor group.  Groups—not individual class rows—are split into training and validation sets.


In [ ]:
import copy
import gc
import hashlib
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from scipy.special import logsumexp
from scipy.stats import multivariate_normal, norm
from torch.utils.data import DataLoader, TensorDataset

from utils_dual_hnde import importance_tail_summary
from utils_hnpe import (
    sample_spline_flow,
    spline_flow_log_prob,
    train_spline_flow,
)
from utils_plotting import export_standalone_figure_script

SEED = 19092026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def set_torch_seed(seed):
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

# The default is a practical first Colab run.  Set FAST_MODE=False for the
# paper-quality run.  SMOKE_MODE only checks the complete code path.
SMOKE_MODE = os.environ.get("EX9_MULTICLASS_SMOKE", "0") == "1"
FAST_MODE = os.environ.get("EX9_MULTICLASS_FAST", "1") == "1"
LOAD_IF_AVAILABLE = os.environ.get("EX9_MULTICLASS_LOAD", "0") == "1"

if SMOKE_MODE:
    RUN_TAG = "smoke"
    N_FLOW, N_CLASS = 2_000, 3_000
    FLOW_EPOCHS, CLASS_EPOCHS, ENSEMBLE_SIZE = 2, 3, 1
    N_CONSTRAINT_GROUPS, N_INNER, N_NESTED = 24, 4, 4
    N_DIAGNOSTIC_REFERENCE = 64
elif FAST_MODE:
    RUN_TAG = "fast"
    N_FLOW, N_CLASS = 35_000, 70_000
    FLOW_EPOCHS, CLASS_EPOCHS, ENSEMBLE_SIZE = 12, 24, 2
    N_CONSTRAINT_GROUPS, N_INNER, N_NESTED = 192, 12, 8
    N_DIAGNOSTIC_REFERENCE = 256
else:
    RUN_TAG = "full"
    N_FLOW, N_CLASS = 250_000, 500_000
    FLOW_EPOCHS, CLASS_EPOCHS, ENSEMBLE_SIZE = 50, 70, 4
    N_CONSTRAINT_GROUPS, N_INNER, N_NESTED = 768, 24, 16
    N_DIAGNOSTIC_REFERENCE = 768

N_GRID_REFERENCE = (
    48 if SMOKE_MODE else (128 if FAST_MODE else 256)
)
N_CALIBRATION_CONTEXTS = 12 if SMOKE_MODE else (60 if FAST_MODE else 200)
N_CALIBRATION_SAMPLES = 128 if SMOKE_MODE else (1024 if FAST_MODE else 4096)

MODEL_DIR = Path("models_exercise9_multiclass_v1") / RUN_TAG
FIGURE_SCRIPT_DIR = Path("exercise9_multiclass_figures_scripts") / RUN_TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_SCRIPT_DIR.mkdir(parents=True, exist_ok=True)

FLOW_MODEL_CONFIG = {
    "n_coupling_layers": 10,
    "hidden_features": 512,
    "hidden_layers": 4,
    "spline_num_bins": 16,
    "spline_tail_bound": 5.0,
    "dropout_probability": 0.0,
}
FLOW_TRAINING_CONFIG = {
    "batch_size": 1024,
    "n_epochs": FLOW_EPOCHS,
    "learning_rate": 1.0e-3,
    "min_learning_rate": 1.0e-11,
    "validation_fraction": 0.2,
    "patience": 10,
    "gradient_clip": 5.0,
}
CLASS_MODEL_CONFIG = {
    "hidden_features": 384 if not SMOKE_MODE else 96,
    "hidden_layers": 4 if not SMOKE_MODE else 2,
    "dropout_probability": 0.0,
}
CLASS_TRAINING_CONFIG = {
    "batch_size_groups": 1024 if not SMOKE_MODE else 256,
    "constraint_batch_groups": 12 if not SMOKE_MODE else 6,
    "n_epochs": CLASS_EPOCHS,
    "learning_rate": 1.0e-3,
    "validation_fraction": 0.2,
    "patience": max(8, CLASS_EPOCHS // 3),
    "gradient_clip": 5.0,
    "warmup_epochs": max(1, CLASS_EPOCHS // 5),
    "ramp_epochs": max(1, CLASS_EPOCHS // 5),
}
LAMBDA_NORM_3, LAMBDA_BRIDGE_3 = 0.20, 0.05
LAMBDA_NORM_4, LAMBDA_BRIDGE_4 = 0.15, 0.03

SIMULATOR_SIGMA = np.array([0.95, 0.38, 0.30], dtype=float)
X_OBS = np.array([0.40, 1.35, 0.12], dtype=float)

def export_exercise9_multiclass_figure(fig, script_name):
    path = export_standalone_figure_script(
        fig, script_name=script_name, output_dir=FIGURE_SCRIPT_DIR
    )
    png_path = FIGURE_SCRIPT_DIR / f"{Path(script_name).stem}.png"
    pdf_path = FIGURE_SCRIPT_DIR / f"{Path(script_name).stem}.pdf"
    fig.savefig(png_path, dpi=220, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print("Exported:", path, png_path, pdf_path)
    return path

print(
    f"mode={RUN_TAG}, N_flow={N_FLOW:,}, N_class={N_CLASS:,}, "
    f"ensemble={ENSEMBLE_SIZE}"
)


## The unchanged Gaussian simulator and exact validation densities

We retain Exercise 9 exactly:

$$
\begin{aligned}
  x_1&\sim\mathcal N(\mu+0.8\alpha,0.95^2),\\
  x_2&\sim\mathcal N(0.72\mu^2-0.4\alpha,0.38^2),\\
  x_3&\sim\mathcal N(0.8\cos\mu+0.3\alpha,0.30^2),
\end{aligned}
$$

with $\rho_\mu=0.9\mathcal N(0,1.5^2)+0.1\mathcal N(0,4^2)$ and
$\rho_\alpha=0.9\mathcal N(0,1^2)+0.1\mathcal N(0,3^2)$.

When $\alpha$ is hidden, it can be integrated analytically for validation.  Writing
$b(\mu)=(\mu,0.72\mu^2,0.8\cos\mu)$ and $v=(0.8,-0.4,0.3)$,

$$
  p_m(x\mid\mu)=0.9\,\mathcal N(x;b,\Sigma_\epsilon+vv^T)
  +0.1\,\mathcal N(x;b,\Sigma_\epsilon+9vv^T).
$$

Neither this expression nor the explicit analytic likelihood is passed to a network.


In [ ]:
def _mixture_logpdf(values, core_sigma, broad_sigma):
    values = np.asarray(values, dtype=float)
    return logsumexp(
        np.stack([
            np.log(0.9) + norm.logpdf(values, 0.0, core_sigma),
            np.log(0.1) + norm.logpdf(values, 0.0, broad_sigma),
        ]),
        axis=0,
    )

def design_mu_logpdf(mu):
    return _mixture_logpdf(mu, 1.5, 4.0)

def design_alpha_logpdf(alpha):
    return _mixture_logpdf(alpha, 1.0, 3.0)

def design_logpdf(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    return design_mu_logpdf(theta[:, 0]) + design_alpha_logpdf(theta[:, 1])

def _sample_mixture(n, core_sigma, broad_sigma, rng):
    broad = rng.random(int(n)) < 0.1
    sigma = np.where(broad, broad_sigma, core_sigma)
    return rng.normal(0.0, sigma)

def sample_mu(n, rng):
    return _sample_mixture(n, 1.5, 4.0, rng).astype(np.float32)

def sample_alpha(n, rng):
    return _sample_mixture(n, 1.0, 3.0, rng).astype(np.float32)

def sample_design(n, rng):
    return np.column_stack([sample_mu(n, rng), sample_alpha(n, rng)]).astype(
        np.float32
    )

def simulator_base_mean(mu):
    mu = np.asarray(mu, dtype=float).ravel()
    return np.column_stack([mu, 0.72 * mu**2, 0.8 * np.cos(mu)])

def simulator_mean(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    mu, alpha = theta[:, 0], theta[:, 1]
    return simulator_base_mean(mu) + alpha[:, None] * np.array([0.8, -0.4, 0.3])

def simulate(theta, rng):
    mean = simulator_mean(theta)
    return (mean + rng.normal(size=mean.shape) * SIMULATOR_SIGMA).astype(
        np.float32
    )

def log_likelihood(x, theta):
    """Analytic truth used only in validation cells."""
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    x = np.atleast_2d(np.asarray(x, dtype=float))
    if len(x) == 1 and len(theta) > 1:
        x = np.repeat(x, len(theta), axis=0)
    if len(x) != len(theta):
        raise ValueError("x and theta must have matching rows or one x row.")
    return np.sum(norm.logpdf(x, simulator_mean(theta), SIMULATOR_SIGMA), axis=1)

_NOISE_COV = np.diag(SIMULATOR_SIGMA**2)
_ALPHA_LOADING = np.array([0.8, -0.4, 0.3])
_MARGINAL_COVS = [
    _NOISE_COV + sigma_alpha**2 * np.outer(_ALPHA_LOADING, _ALPHA_LOADING)
    for sigma_alpha in (1.0, 3.0)
]

def marginal_log_likelihood(x, mu):
    """Exact p_m(x|mu) after integrating the design nuisance mixture."""
    mu = np.asarray(mu, dtype=float).ravel()
    x = np.atleast_2d(np.asarray(x, dtype=float))
    if len(x) == 1 and len(mu) > 1:
        x = np.repeat(x, len(mu), axis=0)
    if len(x) != len(mu):
        raise ValueError("x and mu must have matching rows or one x row.")
    residual = x - simulator_base_mean(mu)
    components = [
        np.log(weight)
        + np.atleast_1d(
            multivariate_normal.logpdf(residual, mean=np.zeros(3), cov=cov)
        )
        for weight, cov in zip((0.9, 0.1), _MARGINAL_COVS)
    ]
    return np.atleast_1d(logsumexp(np.stack(components), axis=0))

def normalize_log_curve(log_density, grid):
    log_density = np.asarray(log_density, dtype=float)
    shift = float(np.max(log_density))
    density = np.exp(log_density - shift)
    integral = np.trapezoid(density, grid)
    return density / integral, shift + np.log(integral)

def normalize_log_surface(log_density, x_grid, y_grid):
    log_density = np.asarray(log_density, dtype=float)
    shift = float(np.max(log_density))
    density = np.exp(log_density - shift)
    integral = np.trapezoid(np.trapezoid(density, y_grid, axis=1), x_grid)
    return density / integral, shift + np.log(integral)

def integrated_absolute_error(reference, estimate, grid):
    return float(np.trapezoid(np.abs(reference - estimate), grid))

def js_distance_discrete(reference, estimate, floor=1.0e-15):
    reference = np.asarray(reference, dtype=float).ravel() + floor
    estimate = np.asarray(estimate, dtype=float).ravel() + floor
    reference /= reference.sum()
    estimate /= estimate.sum()
    middle = 0.5 * (reference + estimate)
    divergence = 0.5 * np.sum(reference * np.log(reference / middle))
    divergence += 0.5 * np.sum(estimate * np.log(estimate / middle))
    return float(np.sqrt(max(0.0, divergence)))

print("Observed x:", X_OBS)
print("Truth check, log p_m(x_obs|mu=1.2):", marginal_log_likelihood(X_OBS, [1.2])[0])


## Shared-logit multiclass trainer

The trainer below is intentionally local to this alternative exercise.  It:

- splits complete triplets or quartets before expanding class rows;
- fits its input standardizer on training groups only;
- preserves equal empirical class priors in every minibatch;
- freezes all flow proposals and differentiates only through the shared logits;
- warms up with CE before smoothly turning on the finite-normalization and bridge penalties;
- averages centered logits across ensemble members, retaining one transitive logit system.

Centered-logit averaging is a geometric-mean ratio ensemble.  The constraints are optimized member by member, so they do not automatically transfer to the ensemble.  Every ensemble partition shown below is therefore re-estimated explicitly from fresh draws; a paper analysis should also show between-member bands.


In [ ]:
class SharedLogitMLP(nn.Module):
    def __init__(
        self,
        input_dim,
        n_classes,
        hidden_features=384,
        hidden_layers=4,
        dropout_probability=0.0,
    ):
        super().__init__()
        layers = []
        width_in = int(input_dim)
        for _ in range(int(hidden_layers)):
            layers.extend([
                nn.Linear(width_in, int(hidden_features)),
                nn.SiLU(),
            ])
            if dropout_probability:
                layers.append(nn.Dropout(float(dropout_probability)))
            width_in = int(hidden_features)
        layers.append(nn.Linear(width_in, int(n_classes)))
        self.network = nn.Sequential(*layers)

    def forward(self, values):
        return self.network(values)

def _assert_finite(name, values, ndim=None):
    values = np.asarray(values)
    if ndim is not None and values.ndim != ndim:
        raise ValueError(f"{name} must have ndim={ndim}; got {values.shape}.")
    if not np.isfinite(values).all():
        raise ValueError(f"{name} contains non-finite values.")
    return values

def _draw_conditional(flow_pack, contexts, n_samples, seed):
    contexts = np.atleast_2d(np.asarray(contexts, dtype=np.float32))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    draws = sample_spline_flow(
        flow_pack, int(n_samples), context=contexts, batch_size=16_384
    )
    draws = np.asarray(draws, dtype=np.float32)
    if len(contexts) == 1:
        draws = draws[None, :, :]
    expected = (len(contexts), int(n_samples), draws.shape[-1])
    if draws.shape != expected:
        raise RuntimeError(f"Unexpected conditional sample shape {draws.shape}; expected {expected}.")
    return draws

def _logmeanexp_torch(values, dim=-1):
    return torch.logsumexp(values, dim=dim) - math.log(values.shape[dim])

def _standardize_points(points, mean, std):
    points = np.asarray(points, dtype=np.float32)
    return ((points - mean) / std).astype(np.float32)

def _prepare_constraint_bundle(bundle, mean, std):
    prepared = {}
    for key, value in bundle.items():
        array = np.asarray(value, dtype=np.float32)
        if key.endswith("_points"):
            array = _standardize_points(array, mean, std)
        prepared[key] = torch.as_tensor(array, dtype=torch.float32, device=device)
    return prepared

def _constraint_scale(epoch, config):
    warmup = int(config["warmup_epochs"])
    ramp = int(config["ramp_epochs"])
    if epoch < warmup:
        return 0.0
    return min(1.0, (epoch - warmup + 1) / max(1, ramp))

def _make_model(input_dim, n_classes):
    return SharedLogitMLP(
        input_dim=input_dim,
        n_classes=n_classes,
        **CLASS_MODEL_CONFIG,
    ).to(device)

def _multiclass_fingerprint(
    class_points,
    *,
    n_classes,
    seed_base,
    lambda_norm,
    lambda_bridge,
    constraint_train,
    constraint_validation,
):
    digest = hashlib.sha256()
    configuration = json.dumps(
        {
            "n_classes": int(n_classes),
            "seed_base": int(seed_base),
            "ensemble_size": int(ENSEMBLE_SIZE),
            "model": CLASS_MODEL_CONFIG,
            "training": CLASS_TRAINING_CONFIG,
            "lambda_norm": float(lambda_norm),
            "lambda_bridge": float(lambda_bridge),
        },
        sort_keys=True,
        separators=(",", ":"),
    )
    digest.update(configuration.encode("utf-8"))
    for name, value in [("class_points", class_points)]:
        array = np.ascontiguousarray(value)
        digest.update(name.encode("utf-8"))
        digest.update(str(array.shape).encode("ascii"))
        digest.update(array.view(np.uint8))
    for bundle_name, bundle in [
        ("constraint_train", constraint_train),
        ("constraint_validation", constraint_validation),
    ]:
        if bundle is None:
            digest.update(f"{bundle_name}:none".encode("utf-8"))
            continue
        for key in sorted(bundle):
            array = np.ascontiguousarray(bundle[key])
            digest.update(f"{bundle_name}:{key}".encode("utf-8"))
            digest.update(str(array.shape).encode("ascii"))
            digest.update(array.view(np.uint8))
    return digest.hexdigest()

def _validation_ce(model, points_tensor, n_classes, batch_groups=2048):
    model.eval()
    total, count = 0.0, 0
    labels_one = torch.arange(n_classes, device=device)
    with torch.no_grad():
        for start in range(0, len(points_tensor), batch_groups):
            batch = points_tensor[start : start + batch_groups].to(device)
            logits = model(batch.reshape(-1, batch.shape[-1]))
            labels = labels_one.repeat(len(batch))
            total += float(F.cross_entropy(logits, labels, reduction="sum").cpu())
            count += len(labels)
    return total / max(1, count)

def train_multiclass_ensemble(
    class_points,
    *,
    n_classes,
    checkpoint_dir,
    seed_base,
    constraint_train=None,
    constraint_validation=None,
    constraint_loss_fn=None,
    lambda_norm=0.0,
    lambda_bridge=0.0,
):
    class_points = _assert_finite("class_points", class_points, ndim=3).astype(np.float32)
    if class_points.shape[1] != n_classes:
        raise ValueError("class_points must contain exactly one row per class per group.")
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    fingerprint = _multiclass_fingerprint(
        class_points,
        n_classes=n_classes,
        seed_base=seed_base,
        lambda_norm=lambda_norm,
        lambda_bridge=lambda_bridge,
        constraint_train=constraint_train,
        constraint_validation=constraint_validation,
    )

    split_rng = np.random.default_rng(SEED + 700 + n_classes)
    order = split_rng.permutation(len(class_points))
    n_validation = max(1, int(CLASS_TRAINING_CONFIG["validation_fraction"] * len(order)))
    validation_index, training_index = order[:n_validation], order[n_validation:]
    if set(validation_index).intersection(set(training_index)):
        raise RuntimeError("Group split leakage detected.")

    training_flat = class_points[training_index].reshape(-1, class_points.shape[-1])
    mean = training_flat.mean(axis=0, dtype=np.float64).astype(np.float32)
    std = training_flat.std(axis=0, dtype=np.float64).astype(np.float32)
    std = np.where(std > 1.0e-6, std, 1.0).astype(np.float32)
    train_tensor = torch.as_tensor(
        _standardize_points(class_points[training_index], mean, std),
        dtype=torch.float32,
    )
    validation_tensor = torch.as_tensor(
        _standardize_points(class_points[validation_index], mean, std),
        dtype=torch.float32,
    )
    train_dataset = TensorDataset(train_tensor)

    prepared_train = None
    prepared_validation = None
    if constraint_loss_fn is not None:
        prepared_train = _prepare_constraint_bundle(constraint_train, mean, std)
        prepared_validation = _prepare_constraint_bundle(constraint_validation, mean, std)

    ensemble = []
    labels_one = torch.arange(n_classes, device=device)
    for member in range(ENSEMBLE_SIZE):
        checkpoint = checkpoint_dir / f"member_{member}.pt"
        if LOAD_IF_AVAILABLE and checkpoint.exists():
            try:
                saved = torch.load(checkpoint, map_location=device, weights_only=False)
            except TypeError:
                saved = torch.load(checkpoint, map_location=device)
            if saved.get("fingerprint") != fingerprint:
                raise RuntimeError(
                    f"Checkpoint {checkpoint} belongs to different data, "
                    "architecture, loss, or constraint draws.  Use a new "
                    "run tag or retrain with EX9_MULTICLASS_LOAD=0."
                )
            model = _make_model(class_points.shape[-1], n_classes)
            model.load_state_dict(saved["state_dict"])
            model.eval()
            ensemble.append({
                "model": model,
                "mean": np.asarray(saved["mean"], dtype=np.float32),
                "std": np.asarray(saved["std"], dtype=np.float32),
                "history": saved.get("history", {}),
                "checkpoint": checkpoint,
            })
            print("Loaded", checkpoint)
            continue

        member_seed = int(seed_base + member)
        torch.manual_seed(member_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(member_seed)
        data_generator = torch.Generator()
        data_generator.manual_seed(member_seed + 10_000)
        constraint_generator = torch.Generator()
        constraint_generator.manual_seed(member_seed + 20_000)
        train_loader = DataLoader(
            train_dataset,
            batch_size=int(CLASS_TRAINING_CONFIG["batch_size_groups"]),
            shuffle=True,
            drop_last=False,
            generator=data_generator,
        )
        model = _make_model(class_points.shape[-1], n_classes)
        optimizer = torch.optim.Adam(model.parameters(), lr=CLASS_TRAINING_CONFIG["learning_rate"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.2, patience=max(2, CLASS_EPOCHS // 8)
        )
        history = {key: [] for key in [
            "train_ce", "validation_ce", "validation_norm",
            "validation_bridge", "validation_total", "constraint_scale", "learning_rate",
        ]}
        best_state, best_value, stale = None, math.inf, 0

        for epoch in range(CLASS_EPOCHS):
            model.train()
            scale = _constraint_scale(epoch, CLASS_TRAINING_CONFIG)
            ce_sum, n_rows = 0.0, 0
            for (group_batch,) in train_loader:
                group_batch = group_batch.to(device)
                logits = model(group_batch.reshape(-1, group_batch.shape[-1]))
                labels = labels_one.repeat(len(group_batch))
                ce = F.cross_entropy(logits, labels)
                norm_loss = torch.zeros((), device=device)
                bridge_loss = torch.zeros((), device=device)
                if constraint_loss_fn is not None and scale > 0.0:
                    n_groups = next(iter(prepared_train.values())).shape[0]
                    index = torch.randint(
                        n_groups,
                        (min(int(CLASS_TRAINING_CONFIG["constraint_batch_groups"]), n_groups),),
                        generator=constraint_generator,
                    ).to(device)
                    norm_loss, bridge_loss = constraint_loss_fn(model, prepared_train, index)
                objective = ce + scale * (
                    float(lambda_norm) * norm_loss + float(lambda_bridge) * bridge_loss
                )
                if not torch.isfinite(objective):
                    raise FloatingPointError("Non-finite multiclass objective.")
                optimizer.zero_grad(set_to_none=True)
                objective.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLASS_TRAINING_CONFIG["gradient_clip"])
                optimizer.step()
                ce_sum += float(ce.detach().cpu()) * len(labels)
                n_rows += len(labels)

            validation_ce = _validation_ce(model, validation_tensor, n_classes)
            validation_norm = 0.0
            validation_bridge = 0.0
            if constraint_loss_fn is not None:
                model.eval()
                with torch.no_grad():
                    n_groups = next(iter(prepared_validation.values())).shape[0]
                    all_index = torch.arange(n_groups, device=device)
                    val_norm_tensor, val_bridge_tensor = constraint_loss_fn(
                        model, prepared_validation, all_index
                    )
                    validation_norm = float(val_norm_tensor.cpu())
                    validation_bridge = float(val_bridge_tensor.cpu())
            validation_total = validation_ce + scale * (
                float(lambda_norm) * validation_norm
                + float(lambda_bridge) * validation_bridge
            )
            if constraint_loss_fn is None or scale >= 1.0:
                scheduler.step(validation_total)
            history["train_ce"].append(ce_sum / max(1, n_rows))
            history["validation_ce"].append(validation_ce)
            history["validation_norm"].append(validation_norm)
            history["validation_bridge"].append(validation_bridge)
            history["validation_total"].append(validation_total)
            history["constraint_scale"].append(scale)
            history["learning_rate"].append(optimizer.param_groups[0]["lr"])

            selection_active = constraint_loss_fn is None or scale >= 1.0
            if selection_active and validation_total < best_value - 1.0e-5:
                best_value = validation_total
                best_state = copy.deepcopy(model.state_dict())
                stale = 0
            elif selection_active:
                stale += 1
            if epoch == 0 or (epoch + 1) % max(1, CLASS_EPOCHS // 6) == 0:
                print(
                    f"member {member}, epoch {epoch + 1:3d}: "
                    f"CE={validation_ce:.4f}, L_Z={validation_norm:.4f}, "
                    f"L_B={validation_bridge:.4f}, ramp={scale:.2f}"
                )
            if stale >= int(CLASS_TRAINING_CONFIG["patience"]) and scale >= 1.0:
                break

        if best_state is None:
            raise RuntimeError("No finite classifier checkpoint was produced.")
        model.load_state_dict(best_state)
        model.eval()
        torch.save({
            "state_dict": model.state_dict(),
            "mean": mean,
            "std": std,
            "history": history,
            "n_classes": n_classes,
            "input_dim": class_points.shape[-1],
            "model_config": CLASS_MODEL_CONFIG,
            "fingerprint": fingerprint,
        }, checkpoint)
        ensemble.append({
            "model": model,
            "mean": mean,
            "std": std,
            "history": history,
            "checkpoint": checkpoint,
        })
        print("Saved", checkpoint)
    return ensemble

@torch.no_grad()
def predict_shared_logits(ensemble, points, batch_size=65_536):
    points = _assert_finite("prediction points", points)
    original_shape = points.shape[:-1]
    flat = points.reshape(-1, points.shape[-1]).astype(np.float32)
    member_logits = []
    for pack in ensemble:
        standardized = _standardize_points(flat, pack["mean"], pack["std"])
        chunks = []
        for start in range(0, len(flat), int(batch_size)):
            tensor = torch.as_tensor(
                standardized[start : start + int(batch_size)],
                dtype=torch.float32,
                device=device,
            )
            logits = pack["model"](tensor)
            logits = logits - logits.mean(dim=1, keepdim=True)
            chunks.append(logits.cpu().numpy())
        member_logits.append(np.concatenate(chunks, axis=0))
    averaged = np.mean(np.stack(member_logits), axis=0)
    return averaged.reshape(*original_shape, averaged.shape[-1])


# Part I — nuisance marginalized implicitly

We first train two frozen proposal flows:

$$q_P(\mu\mid x),\qquad q_L^m(x\mid\mu).$$

Their training simulations still contain $\alpha\sim\rho_\alpha$, but $\alpha$ is omitted from the targets and contexts.  Thus both flows learn the nuisance-marginal problem without ever receiving a nuisance label.

The three balanced classifier classes are

$$
\pi_S=\rho_\mu(\mu)p_m(x\mid\mu),\quad
\pi_P=m_\rho(x)q_P(\mu\mid x),\quad
\pi_L=\rho_\mu(\mu)q_L^m(x\mid\mu).
$$

Consequently $e^{s_S-s_P}$ is the posterior residual and $e^{s_S-s_L}$ is the likelihood residual.  CE sees one simulator row, one posterior-reference row at the same $x$, and one likelihood-reference row at the same $\mu$ in every group.


In [ ]:
# Independent simulations for the two frozen proposals.
rng = np.random.default_rng(SEED + 10)
theta_qp = sample_design(N_FLOW, rng)
x_qp = simulate(theta_qp, rng)
set_torch_seed(SEED + 11)
q_p = train_spline_flow(
    theta_qp[:, :1],
    context=x_qp,
    checkpoint=MODEL_DIR / "q_p_mu_given_x.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 11,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qp, x_qp

rng = np.random.default_rng(SEED + 20)
theta_qlm = sample_design(N_FLOW, rng)
x_qlm = simulate(theta_qlm, rng)
set_torch_seed(SEED + 21)
q_lm = train_spline_flow(
    x_qlm,
    context=theta_qlm[:, :1],
    checkpoint=MODEL_DIR / "q_lm_x_given_mu.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 21,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qlm, x_qlm
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
def build_three_class_groups(n_groups, q_p, q_lm, seed):
    rng = np.random.default_rng(seed)
    theta_s = sample_design(n_groups, rng)
    mu_s = theta_s[:, :1]
    x_s = simulate(theta_s, rng)
    mu_p = _draw_conditional(q_p, x_s, 1, seed + 1)[:, 0, :]
    x_l = _draw_conditional(q_lm, mu_s, 1, seed + 2)[:, 0, :]
    points_s = np.column_stack([mu_s, x_s])
    points_p = np.column_stack([mu_p, x_s])
    points_l = np.column_stack([mu_s, x_l])
    groups = np.stack([points_s, points_p, points_l], axis=1).astype(np.float32)
    return _assert_finite("three-class groups", groups, ndim=3)

three_class_groups = build_three_class_groups(
    N_CLASS, q_p, q_lm, SEED + 100
)
print("Three-class grouped tensor:", three_class_groups.shape)


## What is $\mathcal L_{\rm normalization}$?

It is **not** the normalization of the softmax class probabilities.  The classifier returns residual density ratios; those ratios must themselves integrate to one against the corresponding proposal for every conditioning value:

$$
Z_P(x)=\mathbb E_{\mu\sim q_P(\cdot\mid x)}
  \left[e^{s_S-s_P}\right],\qquad
Z_L(\mu)=\mathbb E_{x'\sim q_L^m(\cdot\mid\mu)}
  \left[e^{s_S-s_L}\right].
$$

At the population optimum both are one.  With multiple frozen proposal draws per conditioning value we estimate them by `logmeanexp` and use

$$
  \mathcal L_{\rm normalization}^{(3)}=
  \frac12\mathbb E_x[(\log\widehat Z_P(x))^2]
  +\frac12\mathbb E_\mu[(\log\widehat Z_L(\mu))^2].
$$

Squaring the log treats over- and under-normalization symmetrically.  Gradients pass through the logits, not through the already-frozen flows.

This is a **finite-Monte-Carlo regularizer**, not an unbiased estimator of a population divergence: `logmeanexp`, the square, and the nested bridge estimates introduce finite-reference bias and variance.  Reusing a large bank makes training practical, while a disjoint bank and fresh, larger proposal draws are used for validation.  Increase `N_INNER`, `N_NESTED`, and the number of constraint groups—and repeat with regenerated banks—before interpreting a full paper run.

Normalization alone is not Bayes compatibility.  A finite model implies the **normalized evidence bridge**

$$
\begin{aligned}
  \log \widehat m_{\rm bridge}(x;\mu)={}&
  \log\rho_\mu(\mu)+\log q_L^m(x\mid\mu)-\log q_P(\mu\mid x)
  +s_P-s_L\\
  &+\log\widehat Z_P(x)-\log\widehat Z_L(\mu).
\end{aligned}
$$

It must be independent of $\mu$ at fixed $x$.  We therefore add
$\mathcal L_{\rm bridge}^{(3)}=\mathbb E_x\operatorname{Var}_{\mu\sim q_P}[\log\widehat m_{\rm bridge}]$.
The variance does not determine the evidence constant, which is why both losses are needed.


In [ ]:
def build_three_constraint_bundle(n_groups, n_inner, n_nested, q_p, q_lm, seed):
    rng = np.random.default_rng(seed)

    # Z_P(x): independent held-out x anchors and multiple q_P draws.
    theta_x = sample_design(n_groups, rng)
    x_zp = simulate(theta_x, rng)
    mu_zp = _draw_conditional(q_p, x_zp, n_inner, seed + 1)
    zp_points = np.concatenate(
        [mu_zp, np.repeat(x_zp[:, None, :], n_inner, axis=1)], axis=2
    )

    # Z_L(mu): independent mu anchors and multiple q_L^m draws.
    mu_zl = sample_mu(n_groups, rng)[:, None]
    x_zl = _draw_conditional(q_lm, mu_zl, n_inner, seed + 2)
    zl_points = np.concatenate(
        [np.repeat(mu_zl[:, None, :], n_inner, axis=1), x_zl], axis=2
    )

    # Evidence invariance: for each fixed x, vary mu~q_P.  A nested q_L
    # sample estimates Z_L separately at each candidate mu.
    theta_b = sample_design(n_groups, rng)
    x_b = simulate(theta_b, rng)
    mu_b = _draw_conditional(q_p, x_b, n_inner, seed + 3)
    x_b_repeat = np.repeat(x_b[:, None, :], n_inner, axis=1)
    bridge_points = np.concatenate([mu_b, x_b_repeat], axis=2)
    flat_mu = mu_b.reshape(-1, 1)
    flat_x = x_b_repeat.reshape(-1, 3)
    log_qp = spline_flow_log_prob(q_p, flat_mu, context=flat_x)
    log_ql_at_x = spline_flow_log_prob(q_lm, flat_x, context=flat_mu)
    bridge_base = (
        design_mu_logpdf(flat_mu[:, 0]) + log_ql_at_x - log_qp
    ).reshape(n_groups, n_inner)

    x_nested = _draw_conditional(q_lm, flat_mu, n_nested, seed + 4)
    bridge_zl_points = np.concatenate(
        [
            np.repeat(flat_mu[:, None, :], n_nested, axis=1),
            x_nested,
        ],
        axis=2,
    ).reshape(n_groups, n_inner, n_nested, 4)

    bundle = {
        "zp_points": zp_points,
        "zl_points": zl_points,
        "bridge_points": bridge_points,
        "bridge_base": bridge_base,
        "bridge_zl_points": bridge_zl_points,
    }
    for name, value in bundle.items():
        _assert_finite(name, value)
    return bundle

def three_constraint_loss(model, bundle, index):
    zp = bundle["zp_points"][index]
    logits_zp = model(zp.reshape(-1, zp.shape[-1])).reshape(*zp.shape[:-1], 3)
    log_zp = _logmeanexp_torch(logits_zp[..., 0] - logits_zp[..., 1], dim=1)

    zl = bundle["zl_points"][index]
    logits_zl = model(zl.reshape(-1, zl.shape[-1])).reshape(*zl.shape[:-1], 3)
    log_zl = _logmeanexp_torch(logits_zl[..., 0] - logits_zl[..., 2], dim=1)
    normalization = 0.5 * (log_zp.square().mean() + log_zl.square().mean())

    bridge = bundle["bridge_points"][index]
    logits_bridge = model(bridge.reshape(-1, bridge.shape[-1])).reshape(
        *bridge.shape[:-1], 3
    )
    log_zp_bridge = _logmeanexp_torch(
        logits_bridge[..., 0] - logits_bridge[..., 1], dim=1
    )
    nested = bundle["bridge_zl_points"][index]
    logits_nested = model(nested.reshape(-1, nested.shape[-1])).reshape(
        *nested.shape[:-1], 3
    )
    log_zl_bridge = _logmeanexp_torch(
        logits_nested[..., 0] - logits_nested[..., 2], dim=2
    )
    log_evidence = (
        bundle["bridge_base"][index]
        + logits_bridge[..., 1]
        - logits_bridge[..., 2]
        + log_zp_bridge[:, None]
        - log_zl_bridge
    )
    bridge_loss = log_evidence.var(dim=1, unbiased=False).mean()
    return normalization, bridge_loss

constraints_three_train = build_three_constraint_bundle(
    N_CONSTRAINT_GROUPS, N_INNER, N_NESTED, q_p, q_lm, SEED + 200
)
constraints_three_validation = build_three_constraint_bundle(
    max(48, N_CONSTRAINT_GROUPS // 2),
    N_INNER,
    N_NESTED,
    q_p,
    q_lm,
    SEED + 300,
)


## Controlled bridge ablation: identical data and architecture

Both ensembles see the same triplets, group split, architecture, initialization seeds, CE warm-up, and normalization penalty.  The first has $\lambda_B=0$; the second adds the bridge variance.  Thus their difference can be attributed to the bridge term rather than to normalization.  The default coefficients are $\lambda_Z=0.20$ and $\lambda_B=0.05$ and are exposed in the configuration cell.

The pointwise shared-logit identities hold for both models.  Any difference below must therefore come from learned conditional normalization and parameter invariance, not from the tautological cycle.


In [ ]:
three_no_bridge = train_multiclass_ensemble(
    three_class_groups,
    n_classes=3,
    checkpoint_dir=MODEL_DIR / "three_class_normalization_only",
    seed_base=SEED + 400,
    constraint_train=constraints_three_train,
    constraint_validation=constraints_three_validation,
    constraint_loss_fn=three_constraint_loss,
    lambda_norm=LAMBDA_NORM_3,
    lambda_bridge=0.0,
)

three_full = train_multiclass_ensemble(
    three_class_groups,
    n_classes=3,
    checkpoint_dir=MODEL_DIR / "three_class_normalized_bridge",
    seed_base=SEED + 400,
    constraint_train=constraints_three_train,
    constraint_validation=constraints_three_validation,
    constraint_loss_fn=three_constraint_loss,
    lambda_norm=LAMBDA_NORM_3,
    lambda_bridge=LAMBDA_BRIDGE_3,
)


## Independent closure of the three-class experiment

Fresh proposal draws are used here; none belongs to CE or constraint training.  We compare:

- the posterior route $q_P e^{s_S-s_P}/Z_P$;
- the likelihood route $\rho_\mu q_L^m e^{s_S-s_L}/Z_L$;
- the normalized evidence bridge.

The highlighted bridge region is where the analytic posterior is at least $10^{-4}$ of its peak.  Large excursions far outside posterior support remain visible in the numerical summaries but should not dominate the visual comparison.


In [ ]:
def three_log_zp(ensemble, x_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    mu = _draw_conditional(q_p, x_values, n_reference, seed)
    points = np.concatenate(
        [mu, np.repeat(x_values[:, None, :], n_reference, axis=1)], axis=2
    )
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 0] - logits[..., 1], axis=1) - np.log(n_reference)

def three_log_zl(ensemble, mu_values, n_reference, seed):
    mu_values = np.asarray(mu_values, dtype=np.float32).reshape(-1, 1)
    x = _draw_conditional(q_lm, mu_values, n_reference, seed)
    points = np.concatenate(
        [np.repeat(mu_values[:, None, :], n_reference, axis=1), x], axis=2
    )
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 0] - logits[..., 2], axis=1) - np.log(n_reference)

def three_closure(ensemble, mu_grid, x_observed, n_reference, seed):
    mu_grid = np.asarray(mu_grid, dtype=float)
    x_grid = np.repeat(np.asarray(x_observed)[None, :], len(mu_grid), axis=0)
    points = np.column_stack([mu_grid, x_grid]).astype(np.float32)
    logits = predict_shared_logits(ensemble, points)
    log_qp = spline_flow_log_prob(q_p, mu_grid[:, None], context=x_grid)
    log_ql = spline_flow_log_prob(q_lm, x_grid, context=mu_grid[:, None])
    log_zp = float(three_log_zp(ensemble, x_observed, n_reference, seed)[0])
    log_zl = three_log_zl(ensemble, mu_grid, n_reference, seed + 1)

    log_posterior_route = log_qp + logits[:, 0] - logits[:, 1] - log_zp
    posterior_route, _ = normalize_log_curve(log_posterior_route, mu_grid)
    log_likelihood_route = log_ql + logits[:, 0] - logits[:, 2] - log_zl
    likelihood_posterior, _ = normalize_log_curve(
        design_mu_logpdf(mu_grid) + log_likelihood_route, mu_grid
    )
    log_evidence_bridge = (
        design_mu_logpdf(mu_grid)
        + log_ql
        - log_qp
        + logits[:, 1]
        - logits[:, 2]
        + log_zp
        - log_zl
    )
    return {
        "posterior": posterior_route,
        "likelihood_posterior": likelihood_posterior,
        "log_evidence_bridge": log_evidence_bridge,
        "log_zp": log_zp,
        "log_zl": log_zl,
        "log_ratio_grid": logits[:, 0] - logits[:, 1],
    }

MU_GRID = np.linspace(-3.8, 3.8, 321)
truth_mu, _ = normalize_log_curve(
    design_mu_logpdf(MU_GRID) + marginal_log_likelihood(X_OBS, MU_GRID),
    MU_GRID,
)
MU_EVIDENCE_GRID = np.linspace(-10.0, 10.0, 4001)
_, LOG_EVIDENCE_TRUTH = normalize_log_curve(
    design_mu_logpdf(MU_EVIDENCE_GRID)
    + marginal_log_likelihood(X_OBS, MU_EVIDENCE_GRID),
    MU_EVIDENCE_GRID,
)
q_p_curve, _ = normalize_log_curve(
    spline_flow_log_prob(
        q_p,
        MU_GRID[:, None],
        context=np.repeat(X_OBS[None, :], len(MU_GRID), axis=0),
    ),
    MU_GRID,
)
closure_no_bridge = three_closure(
    three_no_bridge, MU_GRID, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 500
)
closure_full = three_closure(
    three_full, MU_GRID, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 500
)

# Held-out conditional normalizers at many independent contexts.
rng = np.random.default_rng(SEED + 510)
n_context_check = 36 if SMOKE_MODE else (80 if FAST_MODE else 160)
theta_check = sample_design(n_context_check, rng)
x_check = simulate(theta_check, rng)
mu_check = sample_mu(n_context_check, rng)
heldout_norm = {}
for name, ensemble in [("normalization only", three_no_bridge), ("normalized bridge", three_full)]:
    heldout_norm[name] = {
        "log_zp": three_log_zp(
            ensemble, x_check, N_DIAGNOSTIC_REFERENCE, SEED + 520
        ),
        "log_zl": three_log_zl(
            ensemble, mu_check, N_DIAGNOSTIC_REFERENCE, SEED + 521
        ),
    }

# Importance-tail diagnostics at x_obs.
mu_tail = _draw_conditional(
    q_p, X_OBS[None, :],
    2_000 if SMOKE_MODE else (20_000 if FAST_MODE else 80_000),
    SEED + 530,
)[0]
x_tail = np.repeat(X_OBS[None, :], len(mu_tail), axis=0)
tail_points = np.column_stack([mu_tail, x_tail])
tail_summaries = {}
tail_log_weights = {}
for name, ensemble in [("normalization only", three_no_bridge), ("normalized bridge", three_full)]:
    logits = predict_shared_logits(ensemble, tail_points)
    log_weights = logits[:, 0] - logits[:, 1]
    tail_log_weights[name] = log_weights
    tail_summaries[name] = importance_tail_summary(log_weights)

mu_likelihood_tail = float(MU_GRID[np.argmax(truth_mu)])
x_likelihood_tail = _draw_conditional(
    q_lm,
    [[mu_likelihood_tail]],
    len(mu_tail),
    SEED + 531,
)[0]
likelihood_tail_points = np.column_stack([
    np.full(len(x_likelihood_tail), mu_likelihood_tail),
    x_likelihood_tail,
])
likelihood_tail_summaries = {}
for name, ensemble in [("normalization only", three_no_bridge), ("normalized bridge", three_full)]:
    logits = predict_shared_logits(ensemble, likelihood_tail_points)
    likelihood_tail_summaries[name] = importance_tail_summary(
        logits[:, 0] - logits[:, 2]
    )

rows = []
for name, closure in [("normalization only", closure_no_bridge), ("normalized bridge", closure_full)]:
    normalizers = heldout_norm[name]
    bridge_mask = truth_mu > 1.0e-4 * truth_mu.max()
    bridge_delta = closure["log_evidence_bridge"] - LOG_EVIDENCE_TRUTH
    tails = tail_summaries[name]
    all_log_z = np.concatenate([normalizers["log_zp"], normalizers["log_zl"]])
    rows.append({
        "objective": name,
        "posterior IAE": integrated_absolute_error(truth_mu, closure["posterior"], MU_GRID),
        "likelihood-route IAE": integrated_absolute_error(
            truth_mu, closure["likelihood_posterior"], MU_GRID
        ),
        "bridge RMS (supported)": float(np.sqrt(np.mean(bridge_delta[bridge_mask] ** 2))),
        "RMS log Z": float(np.sqrt(np.mean(all_log_z**2))),
        "q95 |log Z|": float(np.quantile(np.abs(all_log_z), 0.95)),
        "ESS fraction": tails["ESS_fraction"],
        "Pareto k": tails["pareto_k"],
        "max weight": tails["max_weight_fraction"],
        "likelihood ESS fraction": likelihood_tail_summaries[name]["ESS_fraction"],
        "likelihood Pareto k": likelihood_tail_summaries[name]["pareto_k"],
        "likelihood max weight": likelihood_tail_summaries[name]["max_weight_fraction"],
    })
three_summary = pd.DataFrame(rows)
display(three_summary.style.format(precision=4))


In [ ]:
NO_BRIDGE_COLOR = "#D55E00"
FULL3_COLOR = "#0072B2"
FULL4_COLOR = "#009E73"

fig, axes = plt.subplots(2, 2, figsize=(12.2, 8.8), constrained_layout=True)

axes[0, 0].plot(MU_GRID, truth_mu, color="black", lw=2.4, label="analytic truth")
axes[0, 0].plot(MU_GRID, q_p_curve, color="0.6", lw=1.5, ls=":", label=r"proposal $q_P$")
axes[0, 0].plot(
    MU_GRID, closure_no_bridge["posterior"], color=NO_BRIDGE_COLOR, lw=2.0,
    label="no bridge: CE + normalization",
)
axes[0, 0].plot(
    MU_GRID, closure_full["posterior"], color=FULL3_COLOR, lw=2.2,
    label="constrained posterior route",
)
axes[0, 0].plot(
    MU_GRID, closure_full["likelihood_posterior"], color=FULL3_COLOR,
    lw=1.8, ls="--", label="constrained likelihood route",
)
axes[0, 0].set(xlabel=r"$\mu$", ylabel="posterior density", title="(a) Posterior closure")
axes[0, 0].legend(fontsize=8.5)

supported = truth_mu > 1.0e-4 * truth_mu.max()
for label, closure, color in [
    ("CE + normalization", closure_no_bridge, NO_BRIDGE_COLOR),
    ("CE + normalization + bridge", closure_full, FULL3_COLOR),
]:
    residual = closure["log_evidence_bridge"] - LOG_EVIDENCE_TRUTH
    axes[0, 1].plot(
        MU_GRID, np.where(supported, residual, np.nan),
        color=color, lw=2, label=label,
    )
axes[0, 1].axhline(0.0, color="black", lw=1)
axes[0, 1].set(
    xlabel=r"$\mu$", ylabel=r"$\log\widehat m_{\rm bridge}-\log m_{\rm true}$",
    title="(b) Normalized evidence invariance\n(posterior-supported region)",
)
axes[0, 1].legend(fontsize=8.5)

jitter = {"normalization only": -0.06, "normalized bridge": 0.06}
color_map = {"normalization only": NO_BRIDGE_COLOR, "normalized bridge": FULL3_COLOR}
for name in ["normalization only", "normalized bridge"]:
    values_p = heldout_norm[name]["log_zp"]
    values_l = heldout_norm[name]["log_zl"]
    rng_plot = np.random.default_rng(SEED + (1 if name == "normalization only" else 2))
    axes[1, 0].scatter(
        np.full_like(values_p, 0.0 + jitter[name]) + rng_plot.normal(0, 0.012, len(values_p)),
        values_p, s=12, alpha=0.45, color=color_map[name],
    )
    axes[1, 0].scatter(
        np.full_like(values_l, 1.0 + jitter[name]) + rng_plot.normal(0, 0.012, len(values_l)),
        values_l, s=12, alpha=0.45, color=color_map[name], label=name,
    )
axes[1, 0].axhline(0.0, color="black", lw=1)
axes[1, 0].set(
    xticks=[0, 1], xticklabels=[r"$\log Z_P(x)$", r"$\log Z_L(\mu)$"],
    ylabel="held-out conditional log normalizer",
    title="(c) Independent normalization closure",
)
axes[1, 0].legend(fontsize=8.5)

for name, color in [("normalization only", NO_BRIDGE_COLOR), ("normalized bridge", FULL3_COLOR)]:
    log_w = tail_log_weights[name]
    weights = np.exp(log_w - logsumexp(log_w)) * len(log_w)
    ordered = np.sort(np.maximum(weights, np.finfo(float).tiny))
    survival = 1.0 - np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
    axes[1, 1].plot(ordered, survival, color=color, lw=1.8, label=name)
axes[1, 1].set(
    xscale="log", yscale="log", xlabel=r"normalized correction $r_P/\overline r_P$",
    ylabel="empirical survival", title="(d) Posterior-correction tail",
)
axes[1, 1].legend(fontsize=8.5)
for ax in axes.flat:
    ax.grid(alpha=0.25)

export_exercise9_multiclass_figure(fig, "three_class_no_bridge_vs_full_bridge")
plt.show()


**Figure 1 interpretation.**  The two classifiers share the same proposals, triplets, group split, architecture, initialization seeds, and normalization penalty; only the bridge coefficient changes from zero to its configured value.  Panel (b) uses the finite, independently normalized bridge, not a pointwise logit cycle.  The scatter in panel (c) uses fresh proposal draws and therefore includes Monte Carlo error.  Fast-mode curves are pedagogical previews; use full mode before exporting a paper result.


## Held-out simulation-based calibration of the bridge ablation

Closure at one $x_{\rm obs}$ is not evidence of amortized calibration.  We therefore draw an entirely new set $(\mu_i,\alpha_i,x_i)$ from the design, sample $\mu\sim q_P(\cdot\mid x_i)$, and compute posterior PIT values before and after residual weighting.  For a calibrated design posterior these PIT values are uniform, and equal-tailed credible intervals have nominal coverage.

This is a held-out aggregate test, but one training seed is still only one experiment.  For a paper result, repeat the full training and show the between-seed spread around these curves.


In [ ]:
rng = np.random.default_rng(SEED + 1300)
theta_calibration = sample_design(N_CALIBRATION_CONTEXTS, rng)
x_calibration = simulate(theta_calibration, rng)
mu_calibration = _draw_conditional(
    q_p, x_calibration, N_CALIBRATION_SAMPLES, SEED + 1301
)[..., 0]
calibration_points = np.concatenate([
    mu_calibration[..., None],
    np.repeat(x_calibration[:, None, :], N_CALIBRATION_SAMPLES, axis=1),
], axis=2)
below_truth = mu_calibration <= theta_calibration[:, 0, None]

pit_values = {"proposal q_P": below_truth.mean(axis=1)}
for name, ensemble in [
    ("normalization only", three_no_bridge),
    ("normalization + bridge", three_full),
]:
    logits = predict_shared_logits(ensemble, calibration_points)
    log_weights = logits[..., 0] - logits[..., 1]
    weights = np.exp(log_weights - logsumexp(log_weights, axis=1, keepdims=True))
    pit_values[name] = np.sum(weights * below_truth, axis=1)

NOMINAL_COVERAGE = np.linspace(0.05, 0.95, 19)
coverage_values = {
    name: np.array([
        np.mean((pit >= 0.5 * (1.0 - level)) & (pit <= 0.5 * (1.0 + level)))
        for level in NOMINAL_COVERAGE
    ])
    for name, pit in pit_values.items()
}
calibration_rows = []
for name, pit in pit_values.items():
    ordered = np.sort(pit)
    uniform_quantiles = (np.arange(len(ordered)) + 0.5) / len(ordered)
    calibration_rows.append({
        "method": name,
        "PIT KS distance": float(np.max(np.abs(ordered - uniform_quantiles))),
        "max coverage error": float(np.max(np.abs(coverage_values[name] - NOMINAL_COVERAGE))),
    })
calibration_summary = pd.DataFrame(calibration_rows)
display(calibration_summary.style.format(precision=4))

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.3), constrained_layout=True)
calibration_colors = {
    "proposal q_P": "0.55",
    "normalization only": NO_BRIDGE_COLOR,
    "normalization + bridge": FULL3_COLOR,
}
for name, pit in pit_values.items():
    ordered = np.sort(pit)
    empirical_cdf = np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
    axes[0].plot(ordered, empirical_cdf, lw=1.9, color=calibration_colors[name], label=name)
    axes[1].plot(
        NOMINAL_COVERAGE, coverage_values[name], lw=1.9,
        color=calibration_colors[name], label=name,
    )
axes[0].plot([0, 1], [0, 1], color="black", ls="--", lw=1)
axes[0].set(
    xlabel="posterior PIT", ylabel="empirical CDF",
    title="(a) Held-out posterior PIT",
)
axes[1].plot([0, 1], [0, 1], color="black", ls="--", lw=1)
binomial_sigma = np.sqrt(
    NOMINAL_COVERAGE * (1.0 - NOMINAL_COVERAGE) / N_CALIBRATION_CONTEXTS
)
axes[1].fill_between(
    NOMINAL_COVERAGE,
    np.maximum(0.0, NOMINAL_COVERAGE - binomial_sigma),
    np.minimum(1.0, NOMINAL_COVERAGE + binomial_sigma),
    color="0.7", alpha=0.2, linewidth=0, label=r"$\pm1\sigma$ binomial",
)
axes[1].set(
    xlabel="nominal equal-tailed coverage", ylabel="empirical coverage",
    title="(b) Held-out coverage",
)
for ax in axes:
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
export_exercise9_multiclass_figure(fig, "three_class_heldout_calibration")
plt.show()


# Part II — the nuisance parameter is explicit

We now train

$$q_N(\alpha\mid x,\mu),\qquad q_L(x\mid\mu,\alpha),$$

while reusing the frozen marginal proposal $q_P(\mu\mid x)$.  The four balanced classes are

$$
\begin{aligned}
\Pi_S &= \rho_\mu\rho_\alpha p(x\mid\mu,\alpha),\\
\Pi_N &= \rho_\mu p_m(x\mid\mu)q_N(\alpha\mid x,\mu),\\
\Pi_P &= m_\rho(x)q_P(\mu\mid x)q_N(\alpha\mid x,\mu),\\
\Pi_L &= \rho_\mu\rho_\alpha q_L(x\mid\mu,\alpha).
\end{aligned}
$$

Their shared logits provide four useful residuals:

$$
  r_P=e^{s_N-s_P},\quad r_N=e^{s_S-s_N},\quad
  r_L=e^{s_S-s_L},\quad r_{PN}=e^{s_S-s_P}.
$$

At the population optimum $r_P$ is independent of $\alpha$; $r_N$ corrects the conditional nuisance posterior; $r_L$ corrects the explicit likelihood; and $r_{PN}$ corrects the factorized joint posterior proposal.


In [ ]:
# q_L^m is no longer needed for nuisance-aware training.  Keep the pack
# usable for rerunning Part I, but release its GPU memory.
q_lm["flow"].to(torch.device("cpu"))
if torch.cuda.is_available():
    torch.cuda.empty_cache()

rng = np.random.default_rng(SEED + 600)
theta_qn = sample_design(N_FLOW, rng)
x_qn = simulate(theta_qn, rng)
set_torch_seed(SEED + 601)
q_n = train_spline_flow(
    theta_qn[:, 1:2],
    context=np.column_stack([x_qn, theta_qn[:, 0]]),
    checkpoint=MODEL_DIR / "q_n_alpha_given_x_mu.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 601,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qn, x_qn

rng = np.random.default_rng(SEED + 610)
theta_ql = sample_design(N_FLOW, rng)
x_ql = simulate(theta_ql, rng)
set_torch_seed(SEED + 611)
q_l = train_spline_flow(
    x_ql,
    context=theta_ql,
    checkpoint=MODEL_DIR / "q_l_x_given_mu_alpha.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 611,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_ql, x_ql
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
def _qn_context(x, mu):
    x = np.atleast_2d(np.asarray(x, dtype=np.float32))
    mu = np.asarray(mu, dtype=np.float32).reshape(-1, 1)
    if len(x) == 1 and len(mu) > 1:
        x = np.repeat(x, len(mu), axis=0)
    if len(x) != len(mu):
        raise ValueError("x and mu rows must match.")
    return np.column_stack([x, mu])

def _draw_joint_reference(x, n_samples, seed):
    x = np.atleast_2d(np.asarray(x, dtype=np.float32))
    mu = _draw_conditional(q_p, x, n_samples, seed)
    x_repeat = np.repeat(x[:, None, :], n_samples, axis=1)
    flat_context = _qn_context(x_repeat.reshape(-1, 3), mu.reshape(-1, 1))
    alpha = _draw_conditional(q_n, flat_context, 1, seed + 1)[:, 0, :]
    alpha = alpha.reshape(len(x), n_samples, 1)
    return mu, alpha

def build_four_class_groups(n_groups, q_p, q_n, q_l, seed):
    rng = np.random.default_rng(seed)
    theta_s = sample_design(n_groups, rng)
    mu_s, alpha_s = theta_s[:, :1], theta_s[:, 1:2]
    x_s = simulate(theta_s, rng)

    alpha_n = _draw_conditional(q_n, _qn_context(x_s, mu_s), 1, seed + 1)[:, 0, :]
    mu_p = _draw_conditional(q_p, x_s, 1, seed + 2)[:, 0, :]
    alpha_p = _draw_conditional(q_n, _qn_context(x_s, mu_p), 1, seed + 3)[:, 0, :]
    x_l = _draw_conditional(q_l, theta_s, 1, seed + 4)[:, 0, :]

    points_s = np.column_stack([mu_s, alpha_s, x_s])
    points_n = np.column_stack([mu_s, alpha_n, x_s])
    points_p = np.column_stack([mu_p, alpha_p, x_s])
    points_l = np.column_stack([mu_s, alpha_s, x_l])
    groups = np.stack([points_s, points_n, points_p, points_l], axis=1).astype(
        np.float32
    )
    return _assert_finite("four-class groups", groups, ndim=3)

four_class_groups = build_four_class_groups(
    N_CLASS, q_p, q_n, q_l, SEED + 620
)
print("Four-class grouped tensor:", four_class_groups.shape)


## Full nuisance-aware loss

The four conditional partitions are

$$
\begin{aligned}
  Z_P(x)&=\mathbb E_{q_Pq_N}[e^{s_N-s_P}], &
  Z_N(x,\mu)&=\mathbb E_{q_N}[e^{s_S-s_N}],\\
  Z_L(\mu,\alpha)&=\mathbb E_{q_L}[e^{s_S-s_L}], &
  Z_{PN}(x)&=\mathbb E_{q_Pq_N}[e^{s_S-s_P}].
\end{aligned}
$$

$\mathcal L_{\rm normalization}^{(4)}$ is the average squared log of these four Monte Carlo partitions.  The bridge loss contains two physical invariances:

1. the normalized nuisance-marginal likelihood
   $$
   \log\widehat p_m(x\mid\mu;\alpha)=
   \log\rho_\alpha+\log q_L-\log q_N+s_N-s_L
   +\log Z_N-\log Z_L
   $$
   must be independent of $\alpha$;
2. the normalized evidence
   $$
   \log\widehat m(x;\mu,\alpha)=
   \log\rho_\mu+\log\rho_\alpha+\log q_L-\log q_P-\log q_N
   +s_P-s_L+\log Z_{PN}-\log Z_L
   $$
   must be independent of both $\mu$ and $\alpha$.

We additionally give a small weight inside $\mathcal L_{\rm bridge}^{(4)}$ to the population fact that $s_N-s_P$ is independent of $\alpha$.  The three variance components (nuisance bridge, evidence bridge, posterior-ratio invariance) receive relative weights $(0.4,0.4,0.2)$, and the default outer coefficients are $\lambda_Z=0.15$, $\lambda_B=0.03$.  These are finite-model regularizers, not new probability identities.


In [ ]:
def build_four_constraint_bundle(n_groups, n_inner, n_nested, q_p, q_n, q_l, seed):
    rng = np.random.default_rng(seed)

    # Z_P(x): joint q_P q_N proposals, ratio N/P.
    theta_zp_anchor = sample_design(n_groups, rng)
    x_zp = simulate(theta_zp_anchor, rng)
    mu_zp, alpha_zp = _draw_joint_reference(x_zp, n_inner, seed + 1)
    x_zp_repeat = np.repeat(x_zp[:, None, :], n_inner, axis=1)
    zp_points = np.concatenate([mu_zp, alpha_zp, x_zp_repeat], axis=2)

    # Z_N(x,mu): q_N proposals, ratio S/N.
    theta_zn_anchor = sample_design(n_groups, rng)
    x_zn = simulate(theta_zn_anchor, rng)
    mu_zn = theta_zn_anchor[:, :1]
    alpha_zn = _draw_conditional(
        q_n, _qn_context(x_zn, mu_zn), n_inner, seed + 3
    )
    zn_points = np.concatenate([
        np.repeat(mu_zn[:, None, :], n_inner, axis=1),
        alpha_zn,
        np.repeat(x_zn[:, None, :], n_inner, axis=1),
    ], axis=2)

    # Z_L(mu,alpha): q_L proposals, ratio S/L.
    theta_zl = sample_design(n_groups, rng)
    x_zl = _draw_conditional(q_l, theta_zl, n_inner, seed + 4)
    zl_points = np.concatenate([
        np.repeat(theta_zl[:, None, :], n_inner, axis=1), x_zl
    ], axis=2)

    # Z_PN(x): joint q_P q_N proposals, ratio S/P.
    theta_zpn_anchor = sample_design(n_groups, rng)
    x_zpn = simulate(theta_zpn_anchor, rng)
    mu_zpn, alpha_zpn = _draw_joint_reference(x_zpn, n_inner, seed + 5)
    zpn_points = np.concatenate([
        mu_zpn,
        alpha_zpn,
        np.repeat(x_zpn[:, None, :], n_inner, axis=1),
    ], axis=2)

    # Nuisance-marginal bridge: fixed (x,mu), varying alpha~q_N.
    theta_pm_anchor = sample_design(n_groups, rng)
    x_pm = simulate(theta_pm_anchor, rng)
    mu_pm = theta_pm_anchor[:, :1]
    alpha_pm = _draw_conditional(
        q_n, _qn_context(x_pm, mu_pm), n_inner, seed + 7
    )
    x_pm_repeat = np.repeat(x_pm[:, None, :], n_inner, axis=1)
    mu_pm_repeat = np.repeat(mu_pm[:, None, :], n_inner, axis=1)
    pm_points = np.concatenate([mu_pm_repeat, alpha_pm, x_pm_repeat], axis=2)
    flat_mu_pm = mu_pm_repeat.reshape(-1, 1)
    flat_alpha_pm = alpha_pm.reshape(-1, 1)
    flat_x_pm = x_pm_repeat.reshape(-1, 3)
    flat_theta_pm = np.column_stack([flat_mu_pm, flat_alpha_pm])
    pm_base = (
        design_alpha_logpdf(flat_alpha_pm[:, 0])
        + spline_flow_log_prob(q_l, flat_x_pm, context=flat_theta_pm)
        - spline_flow_log_prob(
            q_n, flat_alpha_pm, context=_qn_context(flat_x_pm, flat_mu_pm)
        )
    ).reshape(n_groups, n_inner)

    alpha_zn_nested = _draw_conditional(
        q_n, _qn_context(x_pm, mu_pm), n_nested, seed + 8
    )
    pm_zn_points = np.concatenate([
        np.repeat(mu_pm[:, None, :], n_nested, axis=1),
        alpha_zn_nested,
        np.repeat(x_pm[:, None, :], n_nested, axis=1),
    ], axis=2)
    x_pm_zl_nested = _draw_conditional(q_l, flat_theta_pm, n_nested, seed + 9)
    pm_zl_points = np.concatenate([
        np.repeat(flat_theta_pm[:, None, :], n_nested, axis=1),
        x_pm_zl_nested,
    ], axis=2).reshape(n_groups, n_inner, n_nested, 5)

    # Evidence bridge: fixed x, varying (mu,alpha)~q_P q_N.
    theta_ev_anchor = sample_design(n_groups, rng)
    x_ev = simulate(theta_ev_anchor, rng)
    mu_ev, alpha_ev = _draw_joint_reference(x_ev, n_inner, seed + 10)
    x_ev_repeat = np.repeat(x_ev[:, None, :], n_inner, axis=1)
    ev_points = np.concatenate([mu_ev, alpha_ev, x_ev_repeat], axis=2)
    flat_mu_ev = mu_ev.reshape(-1, 1)
    flat_alpha_ev = alpha_ev.reshape(-1, 1)
    flat_x_ev = x_ev_repeat.reshape(-1, 3)
    flat_theta_ev = np.column_stack([flat_mu_ev, flat_alpha_ev])
    ev_base = (
        design_mu_logpdf(flat_mu_ev[:, 0])
        + design_alpha_logpdf(flat_alpha_ev[:, 0])
        + spline_flow_log_prob(q_l, flat_x_ev, context=flat_theta_ev)
        - spline_flow_log_prob(q_p, flat_mu_ev, context=flat_x_ev)
        - spline_flow_log_prob(
            q_n, flat_alpha_ev, context=_qn_context(flat_x_ev, flat_mu_ev)
        )
    ).reshape(n_groups, n_inner)

    mu_ev_norm, alpha_ev_norm = _draw_joint_reference(x_ev, n_nested, seed + 12)
    ev_zpn_points = np.concatenate([
        mu_ev_norm,
        alpha_ev_norm,
        np.repeat(x_ev[:, None, :], n_nested, axis=1),
    ], axis=2)
    x_ev_zl_nested = _draw_conditional(q_l, flat_theta_ev, n_nested, seed + 14)
    ev_zl_points = np.concatenate([
        np.repeat(flat_theta_ev[:, None, :], n_nested, axis=1),
        x_ev_zl_nested,
    ], axis=2).reshape(n_groups, n_inner, n_nested, 5)

    bundle = {
        "zp_points": zp_points,
        "zn_points": zn_points,
        "zl_points": zl_points,
        "zpn_points": zpn_points,
        "pm_points": pm_points,
        "pm_base": pm_base,
        "pm_zn_points": pm_zn_points,
        "pm_zl_points": pm_zl_points,
        "ev_points": ev_points,
        "ev_base": ev_base,
        "ev_zpn_points": ev_zpn_points,
        "ev_zl_points": ev_zl_points,
    }
    for name, value in bundle.items():
        _assert_finite(name, value)
        if len(value) != n_groups:
            raise RuntimeError(f"{name} lost its anchor-group axis.")
    return bundle

def four_constraint_loss(model, bundle, index):
    def logits_for(key):
        points = bundle[key][index]
        return model(points.reshape(-1, points.shape[-1])).reshape(
            *points.shape[:-1], 4
        )

    logits_zp = logits_for("zp_points")
    log_zp = _logmeanexp_torch(logits_zp[..., 1] - logits_zp[..., 2], dim=1)
    logits_zn = logits_for("zn_points")
    log_zn = _logmeanexp_torch(logits_zn[..., 0] - logits_zn[..., 1], dim=1)
    logits_zl = logits_for("zl_points")
    log_zl = _logmeanexp_torch(logits_zl[..., 0] - logits_zl[..., 3], dim=1)
    logits_zpn = logits_for("zpn_points")
    log_zpn = _logmeanexp_torch(logits_zpn[..., 0] - logits_zpn[..., 2], dim=1)
    normalization = 0.25 * (
        log_zp.square().mean()
        + log_zn.square().mean()
        + log_zl.square().mean()
        + log_zpn.square().mean()
    )

    logits_pm = logits_for("pm_points")
    logits_pm_zn = logits_for("pm_zn_points")
    log_zn_pm = _logmeanexp_torch(
        logits_pm_zn[..., 0] - logits_pm_zn[..., 1], dim=1
    )
    logits_pm_zl = logits_for("pm_zl_points")
    log_zl_pm = _logmeanexp_torch(
        logits_pm_zl[..., 0] - logits_pm_zl[..., 3], dim=2
    )
    log_pm = (
        bundle["pm_base"][index]
        + logits_pm[..., 1]
        - logits_pm[..., 3]
        + log_zn_pm[:, None]
        - log_zl_pm
    )
    pm_variance = log_pm.var(dim=1, unbiased=False).mean()
    rp_alpha_variance = (
        logits_pm[..., 1] - logits_pm[..., 2]
    ).var(dim=1, unbiased=False).mean()

    logits_ev = logits_for("ev_points")
    logits_ev_zpn = logits_for("ev_zpn_points")
    log_zpn_ev = _logmeanexp_torch(
        logits_ev_zpn[..., 0] - logits_ev_zpn[..., 2], dim=1
    )
    logits_ev_zl = logits_for("ev_zl_points")
    log_zl_ev = _logmeanexp_torch(
        logits_ev_zl[..., 0] - logits_ev_zl[..., 3], dim=2
    )
    log_evidence = (
        bundle["ev_base"][index]
        + logits_ev[..., 2]
        - logits_ev[..., 3]
        + log_zpn_ev[:, None]
        - log_zl_ev
    )
    evidence_variance = log_evidence.var(dim=1, unbiased=False).mean()
    bridge = 0.4 * pm_variance + 0.4 * evidence_variance + 0.2 * rp_alpha_variance
    return normalization, bridge

constraints_four_train = build_four_constraint_bundle(
    N_CONSTRAINT_GROUPS, N_INNER, N_NESTED, q_p, q_n, q_l, SEED + 700
)
constraints_four_validation = build_four_constraint_bundle(
    max(48, N_CONSTRAINT_GROUPS // 2),
    N_INNER,
    N_NESTED,
    q_p,
    q_n,
    q_l,
    SEED + 800,
)


In [ ]:
four_full = train_multiclass_ensemble(
    four_class_groups,
    n_classes=4,
    checkpoint_dir=MODEL_DIR / "four_class_full_normalized_bridge",
    seed_base=SEED + 900,
    constraint_train=constraints_four_train,
    constraint_validation=constraints_four_validation,
    constraint_loss_fn=four_constraint_loss,
    lambda_norm=LAMBDA_NORM_4,
    lambda_bridge=LAMBDA_BRIDGE_4,
)


## Four-class posterior closure

We now validate the direct joint-posterior route

$$
  \widehat p(\mu,\alpha\mid x)
  \propto q_P(\mu\mid x)q_N(\alpha\mid x,\mu)e^{s_S-s_P},
$$

and the independently normalized likelihood route

$$
  \widehat p_L(\mu,\alpha\mid x)
  \propto \rho_\mu\rho_\alpha q_L(x\mid\mu,\alpha)
  e^{s_S-s_L}/Z_L(\mu,\alpha).
$$

The normalizing constant $Z_{PN}(x)$ is needed for absolute evidence, but cancels when a grid posterior is normalized numerically.  The analytic contours remain validation-only.


In [ ]:
def four_log_zl(ensemble, theta_values, n_reference, seed):
    theta_values = np.atleast_2d(np.asarray(theta_values, dtype=np.float32))
    # Stream conditioning contexts.  A paper grid has O(10^4) theta
    # values, so materializing every theta x reference draw at once would
    # create multi-GB ensemble-logit temporaries.
    context_batch = max(1, 65_536 // int(n_reference))
    chunks = []
    for start in range(0, len(theta_values), context_batch):
        theta_chunk = theta_values[start : start + context_batch]
        x = _draw_conditional(
            q_l, theta_chunk, n_reference, seed + start
        )
        points = np.concatenate([
            np.repeat(theta_chunk[:, None, :], n_reference, axis=1), x
        ], axis=2)
        logits = predict_shared_logits(ensemble, points)
        chunks.append(
            logsumexp(logits[..., 0] - logits[..., 3], axis=1)
            - np.log(n_reference)
        )
    return np.concatenate(chunks)

def four_log_zn(ensemble, x_values, mu_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    mu_values = np.asarray(mu_values, dtype=np.float32).reshape(-1, 1)
    if len(x_values) == 1 and len(mu_values) > 1:
        x_values = np.repeat(x_values, len(mu_values), axis=0)
    alpha = _draw_conditional(
        q_n, _qn_context(x_values, mu_values), n_reference, seed
    )
    points = np.concatenate([
        np.repeat(mu_values[:, None, :], n_reference, axis=1),
        alpha,
        np.repeat(x_values[:, None, :], n_reference, axis=1),
    ], axis=2)
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 0] - logits[..., 1], axis=1) - np.log(n_reference)

def four_log_zpn(ensemble, x_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    mu, alpha = _draw_joint_reference(x_values, n_reference, seed)
    points = np.concatenate([
        mu, alpha, np.repeat(x_values[:, None, :], n_reference, axis=1)
    ], axis=2)
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 0] - logits[..., 2], axis=1) - np.log(n_reference)

MU_GRID_2D = np.linspace(-3.5, 3.5, 181 if not SMOKE_MODE else 71)
ALPHA_GRID_2D = np.linspace(-3.2, 3.2, 161 if not SMOKE_MODE else 61)
MU_MESH, ALPHA_MESH = np.meshgrid(MU_GRID_2D, ALPHA_GRID_2D, indexing="ij")
THETA_GRID = np.column_stack([MU_MESH.ravel(), ALPHA_MESH.ravel()])
X_GRID = np.repeat(X_OBS[None, :], len(THETA_GRID), axis=0)
POINTS_GRID = np.column_stack([THETA_GRID, X_GRID]).astype(np.float32)

logits_grid = predict_shared_logits(four_full, POINTS_GRID)
log_qp_grid = spline_flow_log_prob(
    q_p, THETA_GRID[:, :1], context=X_GRID
)
log_qn_grid = spline_flow_log_prob(
    q_n,
    THETA_GRID[:, 1:2],
    context=_qn_context(X_GRID, THETA_GRID[:, :1]),
)
log_joint_direct = log_qp_grid + log_qn_grid + logits_grid[:, 0] - logits_grid[:, 2]
posterior_four_direct, _ = normalize_log_surface(
    log_joint_direct.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)

log_zl_grid = four_log_zl(
    four_full, THETA_GRID, N_GRID_REFERENCE, SEED + 1000
)
log_ql_grid = spline_flow_log_prob(q_l, X_GRID, context=THETA_GRID)
log_joint_likelihood = (
    design_logpdf(THETA_GRID)
    + log_ql_grid
    + logits_grid[:, 0]
    - logits_grid[:, 3]
    - log_zl_grid
)
posterior_four_likelihood, _ = normalize_log_surface(
    log_joint_likelihood.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)

log_joint_truth = design_logpdf(THETA_GRID) + log_likelihood(X_OBS, THETA_GRID)
posterior_truth_2d, log_evidence_grid_truth = normalize_log_surface(
    log_joint_truth.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)
truth_mu_2d = np.trapezoid(posterior_truth_2d, ALPHA_GRID_2D, axis=1)
truth_alpha_2d = np.trapezoid(posterior_truth_2d, MU_GRID_2D, axis=0)
direct_mu_2d = np.trapezoid(posterior_four_direct, ALPHA_GRID_2D, axis=1)
direct_alpha_2d = np.trapezoid(posterior_four_direct, MU_GRID_2D, axis=0)
likelihood_mu_2d = np.trapezoid(posterior_four_likelihood, ALPHA_GRID_2D, axis=1)
likelihood_alpha_2d = np.trapezoid(posterior_four_likelihood, MU_GRID_2D, axis=0)
print(
    "Posterior-grid evidence minus wide 1D truth =",
    f"{log_evidence_grid_truth - LOG_EVIDENCE_TRUTH:+.4f}",
    "(finite plotting-window truncation)",
)
four_summary = pd.DataFrame([
    {
        "route": "direct joint posterior",
        "2D JS distance": js_distance_discrete(
            posterior_truth_2d, posterior_four_direct
        ),
        "mu marginal IAE": integrated_absolute_error(
            truth_mu_2d, direct_mu_2d, MU_GRID_2D
        ),
        "alpha marginal IAE": integrated_absolute_error(
            truth_alpha_2d, direct_alpha_2d, ALPHA_GRID_2D
        ),
    },
    {
        "route": "normalized likelihood",
        "2D JS distance": js_distance_discrete(
            posterior_truth_2d, posterior_four_likelihood
        ),
        "mu marginal IAE": integrated_absolute_error(
            truth_mu_2d, likelihood_mu_2d, MU_GRID_2D
        ),
        "alpha marginal IAE": integrated_absolute_error(
            truth_alpha_2d, likelihood_alpha_2d, ALPHA_GRID_2D
        ),
    },
])
display(four_summary.style.format(precision=4))


In [ ]:
def highest_density_levels(density, x_grid, y_grid, masses=(0.5, 0.9)):
    density = np.asarray(density, dtype=float)
    dx = float(np.mean(np.diff(x_grid)))
    dy = float(np.mean(np.diff(y_grid)))
    ordered = np.sort(density.ravel())[::-1]
    cumulative = np.cumsum(ordered) * dx * dy
    return sorted([
        ordered[min(np.searchsorted(cumulative, mass), len(ordered) - 1)]
        for mass in masses
    ])

truth_levels = highest_density_levels(
    posterior_truth_2d, MU_GRID_2D, ALPHA_GRID_2D
)
direct_levels = highest_density_levels(
    posterior_four_direct, MU_GRID_2D, ALPHA_GRID_2D
)
likelihood_levels = highest_density_levels(
    posterior_four_likelihood, MU_GRID_2D, ALPHA_GRID_2D
)

fig, axes = plt.subplots(2, 2, figsize=(11.8, 9.0), constrained_layout=True)
axes[0, 0].contour(
    MU_GRID_2D, ALPHA_GRID_2D, posterior_truth_2d.T,
    levels=truth_levels, colors="black", linewidths=[2.2, 1.5],
)
axes[0, 0].contour(
    MU_GRID_2D, ALPHA_GRID_2D, posterior_four_direct.T,
    levels=direct_levels, colors=FULL4_COLOR, linewidths=[2.2, 1.5], linestyles="--",
)
axes[0, 0].set_title("(a) Joint posterior: direct route\nblack truth; green learned")

axes[0, 1].contour(
    MU_GRID_2D, ALPHA_GRID_2D, posterior_truth_2d.T,
    levels=truth_levels, colors="black", linewidths=[2.2, 1.5],
)
axes[0, 1].contour(
    MU_GRID_2D, ALPHA_GRID_2D, posterior_four_likelihood.T,
    levels=likelihood_levels, colors=FULL4_COLOR, linewidths=[2.2, 1.5], linestyles="--",
)
axes[0, 1].set_title("(b) Joint posterior: likelihood route\nblack truth; green learned")
for ax in axes[0]:
    ax.set(xlabel=r"$\mu$", ylabel=r"$\alpha$")
    ax.grid(alpha=0.2)

three_full_interp = np.interp(MU_GRID_2D, MU_GRID, closure_full["posterior"])
axes[1, 0].plot(MU_GRID_2D, truth_mu_2d, color="black", lw=2.3, label="analytic truth")
axes[1, 0].plot(
    MU_GRID_2D, three_full_interp, color=FULL3_COLOR, lw=1.8,
    label="three-class (alpha hidden)",
)
axes[1, 0].plot(
    MU_GRID_2D, direct_mu_2d, color=FULL4_COLOR, lw=2.1,
    label="four-class direct",
)
axes[1, 0].plot(
    MU_GRID_2D, likelihood_mu_2d, color=FULL4_COLOR, lw=1.8, ls="--",
    label="four-class likelihood",
)
axes[1, 0].set(xlabel=r"$\mu$", ylabel="marginal posterior", title="(c) Parameter of interest")
axes[1, 0].legend(fontsize=8.5)

axes[1, 1].plot(ALPHA_GRID_2D, truth_alpha_2d, color="black", lw=2.3, label="analytic truth")
axes[1, 1].plot(
    ALPHA_GRID_2D, direct_alpha_2d, color=FULL4_COLOR, lw=2.1,
    label="four-class direct",
)
axes[1, 1].plot(
    ALPHA_GRID_2D, likelihood_alpha_2d, color=FULL4_COLOR, lw=1.8, ls="--",
    label="four-class likelihood",
)
axes[1, 1].set(xlabel=r"$\alpha$", ylabel="marginal posterior", title="(d) Systematic nuisance")
axes[1, 1].legend(fontsize=8.5)
for ax in axes[1]:
    ax.grid(alpha=0.25)

export_exercise9_multiclass_figure(fig, "four_class_joint_posterior_closure")
plt.show()


## The two nuisance-aware bridge invariances

The next figure is the most direct systematic-uncertainty demonstration.  It reconstructs $p_m(x_{\rm obs}\mid\mu)$ at fixed $\mu$ while deliberately varying $\alpha$, and reconstructs $m_\rho(x_{\rm obs})$ along two paths through $(\mu,\alpha)$.  Fresh $q_N$ and $q_L$ samples estimate every displayed partition.

A perfect population classifier gives horizontal zero residuals.  A finite run need not do so; the departures are precisely the quantities the full loss is designed to suppress.


In [ ]:
def nuisance_bridge_curve(ensemble, mu_value, alpha_grid, x_observed, n_reference, seed):
    alpha_grid = np.asarray(alpha_grid, dtype=float)
    theta = np.column_stack([
        np.full_like(alpha_grid, float(mu_value)), alpha_grid
    ]).astype(np.float32)
    x = np.repeat(np.asarray(x_observed)[None, :], len(theta), axis=0).astype(np.float32)
    points = np.column_stack([theta, x])
    logits = predict_shared_logits(ensemble, points)
    log_ql = spline_flow_log_prob(q_l, x, context=theta)
    log_qn = spline_flow_log_prob(
        q_n, theta[:, 1:2], context=_qn_context(x, theta[:, :1])
    )
    log_zn = float(four_log_zn(
        ensemble, x_observed, [mu_value], n_reference, seed
    )[0])
    log_zl = four_log_zl(ensemble, theta, n_reference, seed + 1)
    raw = design_alpha_logpdf(alpha_grid) + log_ql - log_qn + logits[:, 1] - logits[:, 3]
    normalized = raw + log_zn - log_zl
    truth = float(marginal_log_likelihood(x_observed, [mu_value])[0])
    return raw - truth, normalized - truth

def evidence_bridge_path(ensemble, theta, x_observed, n_reference, seed):
    theta = np.atleast_2d(np.asarray(theta, dtype=np.float32))
    x = np.repeat(np.asarray(x_observed)[None, :], len(theta), axis=0).astype(np.float32)
    points = np.column_stack([theta, x])
    logits = predict_shared_logits(ensemble, points)
    log_qp = spline_flow_log_prob(q_p, theta[:, :1], context=x)
    log_qn = spline_flow_log_prob(
        q_n, theta[:, 1:2], context=_qn_context(x, theta[:, :1])
    )
    log_ql = spline_flow_log_prob(q_l, x, context=theta)
    log_zpn = float(four_log_zpn(
        ensemble, x_observed, n_reference, seed
    )[0])
    log_zl = four_log_zl(ensemble, theta, n_reference, seed + 1)
    raw = (
        design_logpdf(theta) + log_ql - log_qp - log_qn
        + logits[:, 2] - logits[:, 3]
    )
    normalized = raw + log_zpn - log_zl
    return raw - LOG_EVIDENCE_TRUTH, normalized - LOG_EVIDENCE_TRUTH

# Locate the two most separated posterior modes for illustrative slices.
local_peak = np.r_[False, (truth_mu[1:-1] > truth_mu[:-2]) & (truth_mu[1:-1] > truth_mu[2:]), False]
peak_candidates = MU_GRID[local_peak]
peak_heights = truth_mu[local_peak]
if len(peak_candidates) >= 2:
    selected = np.argsort(peak_heights)[-2:]
    MU_SLICES = np.sort(peak_candidates[selected])
else:
    MU_SLICES = np.array([-1.4, 1.2])

ALPHA_BRIDGE_GRID = np.linspace(-2.6, 2.6, 81 if not SMOKE_MODE else 31)
nuisance_curves = {}
for index, mu_value in enumerate(MU_SLICES):
    nuisance_curves[float(mu_value)] = nuisance_bridge_curve(
        four_full,
        mu_value,
        ALPHA_BRIDGE_GRID,
        X_OBS,
        N_DIAGNOSTIC_REFERENCE,
        SEED + 1100 + 10 * index,
    )

MU_PATH = np.linspace(-3.0, 3.0, 101 if not SMOKE_MODE else 41)
THETA_MU_PATH = np.column_stack([MU_PATH, np.zeros_like(MU_PATH)])
raw_mu_path, normalized_mu_path = evidence_bridge_path(
    four_full, THETA_MU_PATH, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 1150
)
ALPHA_PATH = np.linspace(-2.6, 2.6, 101 if not SMOKE_MODE else 41)
mu_mode = float(MU_GRID[np.argmax(truth_mu)])
THETA_ALPHA_PATH = np.column_stack([
    np.full_like(ALPHA_PATH, mu_mode), ALPHA_PATH
])
raw_alpha_path, normalized_alpha_path = evidence_bridge_path(
    four_full, THETA_ALPHA_PATH, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 1160
)

# Held-out Z distributions, evaluated directly from the validation bundle.
def bundle_log_normalizers_four(ensemble, bundle):
    logits_zp = predict_shared_logits(ensemble, bundle["zp_points"])
    logits_zn = predict_shared_logits(ensemble, bundle["zn_points"])
    logits_zl = predict_shared_logits(ensemble, bundle["zl_points"])
    logits_zpn = predict_shared_logits(ensemble, bundle["zpn_points"])
    return {
        r"$Z_P$": logsumexp(logits_zp[..., 1] - logits_zp[..., 2], axis=1) - np.log(logits_zp.shape[1]),
        r"$Z_N$": logsumexp(logits_zn[..., 0] - logits_zn[..., 1], axis=1) - np.log(logits_zn.shape[1]),
        r"$Z_L$": logsumexp(logits_zl[..., 0] - logits_zl[..., 3], axis=1) - np.log(logits_zl.shape[1]),
        r"$Z_{PN}$": logsumexp(logits_zpn[..., 0] - logits_zpn[..., 2], axis=1) - np.log(logits_zpn.shape[1]),
    }

diagnostic_four_bundle = build_four_constraint_bundle(
    max(48, n_context_check),
    min(64, N_DIAGNOSTIC_REFERENCE),
    4,
    q_p,
    q_n,
    q_l,
    SEED + 1190,
)
heldout_four_z = bundle_log_normalizers_four(
    four_full, diagnostic_four_bundle
)
for name, values in heldout_four_z.items():
    print(
        f"{name:8s}: RMS(log Z)={np.sqrt(np.mean(values**2)):.3f}, "
        f"q95|log Z|={np.quantile(np.abs(values), .95):.3f}"
    )

n_tail_four = 2_000 if SMOKE_MODE else (20_000 if FAST_MODE else 80_000)
mu_joint_tail, alpha_joint_tail = _draw_joint_reference(
    X_OBS[None, :], n_tail_four, SEED + 1191
)
joint_tail_points = np.concatenate([
    mu_joint_tail,
    alpha_joint_tail,
    np.repeat(X_OBS[None, None, :], n_tail_four, axis=1),
], axis=2)[0]
joint_tail_logits = predict_shared_logits(four_full, joint_tail_points)

alpha_n_tail = _draw_conditional(
    q_n,
    _qn_context(X_OBS, [mu_mode]),
    n_tail_four,
    SEED + 1192,
)[0]
nuisance_tail_points = np.column_stack([
    np.full(n_tail_four, mu_mode),
    alpha_n_tail[:, 0],
    np.repeat(X_OBS[None, :], n_tail_four, axis=0),
])
nuisance_tail_logits = predict_shared_logits(four_full, nuisance_tail_points)

theta_mode = THETA_GRID[int(np.argmax(posterior_truth_2d))]
x_l_tail = _draw_conditional(
    q_l, theta_mode[None, :], n_tail_four, SEED + 1193
)[0]
likelihood_tail_points = np.column_stack([
    np.repeat(theta_mode[None, :], n_tail_four, axis=0), x_l_tail
])
likelihood_tail_logits = predict_shared_logits(four_full, likelihood_tail_points)

four_tail_log_ratios = {
    r"$r_P=N/P$": joint_tail_logits[:, 1] - joint_tail_logits[:, 2],
    r"$r_N=S/N$": nuisance_tail_logits[:, 0] - nuisance_tail_logits[:, 1],
    r"$r_L=S/L$": likelihood_tail_logits[:, 0] - likelihood_tail_logits[:, 3],
    r"$r_{PN}=S/P$": joint_tail_logits[:, 0] - joint_tail_logits[:, 2],
}
four_tail_rows = []
for ratio_name, log_ratio in four_tail_log_ratios.items():
    summary = importance_tail_summary(log_ratio)
    four_tail_rows.append({
        "partition ratio": ratio_name,
        "log mean ratio": float(logsumexp(log_ratio) - np.log(len(log_ratio))),
        "ESS fraction": summary["ESS_fraction"],
        "Pareto k": summary["pareto_k"],
        "max weight": summary["max_weight_fraction"],
    })
four_tail_summary = pd.DataFrame(four_tail_rows)
display(four_tail_summary.style.format(precision=4))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.7), constrained_layout=True)
colors = ["#CC79A7", FULL4_COLOR]
for color, (mu_value, (raw, normalized)) in zip(colors, nuisance_curves.items()):
    axes[0, 0].plot(
        ALPHA_BRIDGE_GRID, raw, color=color, lw=1.2, ls=":",
        label=rf"raw, $\mu={mu_value:+.2f}$",
    )
    axes[0, 0].plot(
        ALPHA_BRIDGE_GRID, normalized, color=color, lw=2.0,
        label=rf"normalized, $\mu={mu_value:+.2f}$",
    )
axes[0, 0].axhline(0.0, color="black", lw=1)
axes[0, 0].set(
    xlabel=r"$\alpha$ used in the reconstruction",
    ylabel=r"$\log\widehat p_m-\log p_m^{\rm true}$",
    title="(a) Nuisance-marginal bridge",
)
axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(MU_PATH, raw_mu_path, color="0.55", lw=1.2, ls=":", label="raw")
axes[0, 1].plot(MU_PATH, normalized_mu_path, color=FULL4_COLOR, lw=2.0, label="normalized")
axes[0, 1].axhline(0.0, color="black", lw=1)
axes[0, 1].set(
    xlabel=r"$\mu$ at $\alpha=0$", ylabel=r"$\log\widehat m-\log m^{\rm true}$",
    title="(b) Evidence bridge along a POI path",
)
axes[0, 1].legend(fontsize=8.5)

axes[1, 0].plot(ALPHA_PATH, raw_alpha_path, color="0.55", lw=1.2, ls=":", label="raw")
axes[1, 0].plot(ALPHA_PATH, normalized_alpha_path, color=FULL4_COLOR, lw=2.0, label="normalized")
axes[1, 0].axhline(0.0, color="black", lw=1)
axes[1, 0].set(
    xlabel=rf"$\alpha$ at $\mu={mu_mode:+.2f}$",
    ylabel=r"$\log\widehat m-\log m^{\rm true}$",
    title="(c) Evidence bridge along a nuisance path",
)
axes[1, 0].legend(fontsize=8.5)

for label, values in heldout_four_z.items():
    ordered = np.sort(values)
    cdf = np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
    axes[1, 1].plot(ordered, cdf, lw=1.8, label=label)
axes[1, 1].axvline(0.0, color="black", lw=1)
axes[1, 1].set(
    xlabel=r"held-out $\log Z$", ylabel="empirical CDF",
    title="(d) Four conditional partitions",
)
axes[1, 1].legend(fontsize=8.5)
for ax in axes.flat:
    ax.grid(alpha=0.25)

export_exercise9_multiclass_figure(fig, "four_class_bridge_and_normalization")
plt.show()


## A systematic-prior and auxiliary-measurement update

Because the four-class model retains $\alpha$, the same trained posterior can be updated without retraining.  We replace the nuisance design density by
$\pi_{\rm new}(\alpha)=\mathcal N(0.30,0.45^2)$ and include an auxiliary measurement
$a_{\rm obs}=0.10$ with $a\mid\alpha\sim\mathcal N(\alpha,0.25^2)$.

On the grid the learned joint density is multiplied by

$$
  \frac{\pi_{\rm new}(\alpha)}{\rho_\alpha(\alpha)}p(a_{\rm obs}\mid\alpha).
$$

This cell is an application check, not part of classifier training.


In [ ]:
ALPHA_PRIOR_MEAN, ALPHA_PRIOR_SIGMA = 0.30, 0.45
A_OBSERVED, SIGMA_A = 0.10, 0.25
log_update_factor = (
    norm.logpdf(
        THETA_GRID[:, 1], loc=ALPHA_PRIOR_MEAN, scale=ALPHA_PRIOR_SIGMA
    )
    - design_alpha_logpdf(THETA_GRID[:, 1])
    + norm.logpdf(A_OBSERVED, loc=THETA_GRID[:, 1], scale=SIGMA_A)
).reshape(MU_MESH.shape)
learned_updated, _ = normalize_log_surface(
    np.log(np.maximum(posterior_four_direct, 1.0e-300)) + log_update_factor,
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
truth_updated, _ = normalize_log_surface(
    log_joint_truth.reshape(MU_MESH.shape) + log_update_factor,
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
learned_update_mu = np.trapezoid(learned_updated, ALPHA_GRID_2D, axis=1)
learned_update_alpha = np.trapezoid(learned_updated, MU_GRID_2D, axis=0)
truth_update_mu = np.trapezoid(truth_updated, ALPHA_GRID_2D, axis=1)
truth_update_alpha = np.trapezoid(truth_updated, MU_GRID_2D, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11.3, 4.3), constrained_layout=True)
axes[0].plot(MU_GRID_2D, truth_update_mu, color="black", lw=2.3, label="updated truth")
axes[0].plot(MU_GRID_2D, learned_update_mu, color=FULL4_COLOR, lw=2.0, label="updated four-class")
axes[0].plot(MU_GRID_2D, truth_mu_2d, color="0.6", ls="--", label="baseline truth")
axes[0].set(xlabel=r"$\mu$", ylabel="posterior density", title="(a) Updated POI")
axes[1].plot(ALPHA_GRID_2D, truth_update_alpha, color="black", lw=2.3, label="updated truth")
axes[1].plot(ALPHA_GRID_2D, learned_update_alpha, color=FULL4_COLOR, lw=2.0, label="updated four-class")
axes[1].plot(ALPHA_GRID_2D, truth_alpha_2d, color="0.6", ls="--", label="baseline truth")
axes[1].set(xlabel=r"$\alpha$", ylabel="posterior density", title="(b) Updated systematic")
for ax in axes:
    ax.legend(fontsize=8.5)
    ax.grid(alpha=0.25)
export_exercise9_multiclass_figure(fig, "four_class_systematic_update")
plt.show()


## Conclusions and paper-run checklist

What this exercise establishes—if the full-run closure plots support it—is more specific than “three logits obey a bridge relation”:

1. A single multiclass discriminator can estimate simulator-anchored posterior and likelihood residuals simultaneously.
2. The pointwise ratio cycle is automatic and is **not** evidence of Bayes consistency.
3. $\mathcal L_{\rm normalization}$ controls the missing conditional partition functions, while $\mathcal L_{\rm bridge}$ controls parameter invariance of the resulting normalized evidence.
4. The four-class extension separates a conditional nuisance correction from the marginal-POI and explicit-likelihood corrections, and gives two physically meaningful invariances.

Before using the figures in the paper:

- rerun with `EX9_MULTICLASS_FAST=0` and a GPU;
- keep `LOAD_IF_AVAILABLE=0` after any change to data, architecture, or loss weights;
- report the held-out $\log Z$ RMS/95th percentile and the posterior ESS/Pareto-$k$, not only the illustrative curves;
- repeat with several seeds or show ensemble/Monte Carlo bands;
- regenerate the constraint banks and increase inner/nested reference counts to quantify finite-Monte-Carlo regularization bias;
- treat “the constrained loss beats CE” as an empirical result, never as a hard assertion;
- use the exported `.py`, `.png`, and `.pdf` files under `exercise9_multiclass_figures_scripts/full/`.

A natural next ablation is a structured-logit architecture that builds some invariances in exactly.  The generic-logit experiment here is deliberately the cleaner test of what the two added losses contribute.
